# Notebook 2: Online Feature Store Setup

Sets up the Postgres-backed Online Service, registers stream feature views with continuous aggregation for all 4 entity velocity groups, and benchmarks freshness and lookup latency.

---

### Architecture

```
FRAUD_TRANSACTIONS (source of truth)
      │
      └──► REST Ingest API ──► Online Feature Store (Postgres)
                                  ├── Stream FVs: CONTINUOUS aggregation (< 2s)
                                  └── Batch FV: Customer profile (daily)
```

### Feature Coverage

| Entity | Stream aggregations | Notes |
|---|---|---|
| Customer | count, sum, max, min, approx_count_distinct — 1h/6h/24h/48h/1wk | Primary card-testing signal |
| Merchant | count, sum, approx_count_distinct — 1h/6h/24h/48h/1wk | Merchant under attack |
| Wallet DPAN | count, sum, approx_count_distinct — 1h/6h/24h/48h/1wk | Compromised card |
| IP Address | count, sum, approx_count_distinct — 1h/6h/24h/48h/1wk | Bot farm |
| Customer Profile | Lifetime stats, account age | Batch online FV, daily refresh |

Derived ratio features (velocity ratios, concentration, deviation) are computed inline in the SPCS scoring container.

> **Preview note:** Requires `snowflake-ml-python >= 1.41` and `SNOWFLAKE_PAT` env var.

In [ ]:
!pip install --upgrade snowflake-ml-python

import importlib
import importlib.metadata
importlib.invalidate_caches()

# The Online Feature Store API requires snowflake-ml-python >= 1.41.
v = importlib.metadata.version('snowflake-ml-python')
major, minor = int(v.split('.')[0]), int(v.split('.')[1])
if (major, minor) >= (1, 41):
    print(f'snowflake-ml-python {v} — OK')
else:
    raise RuntimeError(f'Requires >= 1.41. Found: {v}\nRun: pip install --upgrade snowflake-ml-python')

In [ ]:
import time, os, numpy as np, requests, random, concurrent.futures
from datetime import datetime
from snowflake.snowpark.context import get_active_session

# snowflake.ml.feature_store is the Snowflake Feature Store Python SDK.
# FeatureStore   — top-level handle to a feature store object in Snowflake.
# FeatureView    — a group of features computed from a source table or stream.
# Entity         — a named key (e.g. CUSTOMER_ID) that joins feature views to transactions.
# OnlineConfig   — controls whether a feature view is served from the low-latency online store.
# StreamSource   — defines the Snowflake stream that feeds real-time (CONTINUOUS) feature views.
# Feature        — a named aggregate (count, sum, max, approx_count_distinct) with a time window.
from snowflake.ml.feature_store import (
    FeatureStore, FeatureView, Entity, CreationMode,
    OnlineConfig, OnlineStoreType, StreamSource, StreamConfig, Feature
)
from snowflake.ml.feature_store.spec.enums import FeatureAggregationMethod
from snowflake.ml.feature_store import online_service as ofs_utils

# StructType / StructField define the schema for the StreamSource —
# Snowflake needs to know the column types before the stream is registered.
from snowflake.snowpark.types import (
    StructType, StructField, StringType, FloatType,
    TimestampType, TimestampTimeZone
)

# get_active_session() retrieves the current Snowflake session when running inside
# a Snowflake Notebook or Snowsight. No connection string or credentials needed.
session = get_active_session()

# Switch role and context — FRAUD_MLOPS is the operational role with CREATE FEATURE VIEW privileges.
session.sql('USE ROLE FRAUD_MLOPS').collect()
session.sql('USE DATABASE FRAUD_DEMO_DEV').collect()
session.sql('USE SCHEMA FEATURES').collect()
session.sql('USE WAREHOUSE FRAUD_DEMO_WH').collect()

# Initialise the Feature Store handle.
# creation_mode=CREATE_IF_NOT_EXIST means this is idempotent — safe to re-run.
# The Feature Store is a schema-level object in Snowflake (FRAUD_DEMO_DEV.FEATURE_STORE).
fs = FeatureStore(
    session=session,
    database='FRAUD_DEMO_DEV',
    name='FEATURE_STORE',
    default_warehouse='FRAUD_DEMO_WH',
    creation_mode=CreationMode.CREATE_IF_NOT_EXIST,
)

# PAT (Personal Access Token) is required to call the Online Feature Store HTTP endpoints.
# Set it before running: export SNOWFLAKE_PAT="<your_token>"
# Generate via: Snowsight → Profile → Personal Access Tokens → + New Token
PAT = os.environ.get('SNOWFLAKE_PAT', '')
if not PAT:
    print('WARNING: SNOWFLAKE_PAT not set. Set: export SNOWFLAKE_PAT="<token>"')
else:
    print('PAT: set')
print('Feature Store: FRAUD_DEMO_DEV.FEATURE_STORE')

## 2.1 Create Online Service

Provisions a Postgres-backed serving layer co-located with your Snowflake account. Created once per Feature Store — takes 3-5 minutes on first run.

Exposes REST ingest and query endpoints (public, PrivateLink, or SPCS-internal). For production, configure PrivateLink to keep traffic off the public internet.

In [ ]:
print('Creating Online Service...\nFirst creation takes 3-5 minutes. Polling until RUNNING.\n')

# The Online Service is the Postgres-backed, low-latency serving layer for CONTINUOUS feature views.
# It runs as a managed SPCS (Snowpark Container Services) pod inside your Snowflake account.
# create_online_service(role, warehouse) — the role/warehouse used by the service to read features.
try:
    fs.create_online_service('FRAUD_MLOPS', 'FRAUD_MLOPS')
except Exception as e:
    if 'already exists' in str(e).lower():
        print('Online service already exists — continuing')
    else:
        raise

# Poll until the service reaches RUNNING state.
# PENDING → STARTING → RUNNING typically takes 3-5 minutes on first creation.
# Subsequent runs (service already exists) are instantaneous.
status = fs.get_online_service_status()
start = time.time()
while status.status != 'RUNNING':
    print(f'  [{time.time()-start:.0f}s] {status.status} — waiting 30s...')
    time.sleep(30)
    status = fs.get_online_service_status()

print(f'Online Service RUNNING ({time.time()-start:.0f}s)')

# Build the HTTP headers and base URLs used by all subsequent ingest/query calls.
# The PAT acts as a bearer token — same as using Snowflake OAuth for REST API calls.
HEADERS = {'Authorization': f'Snowflake Token="{PAT}"', 'Content-Type': 'application/json'}
# ofs_utils.endpoint_url() extracts the correct URL from the service status object.
# 'query'  → POST endpoint to retrieve feature values by entity key.
# 'ingest' → POST endpoint to push new event records into the stream.
QUERY_URL  = ofs_utils.endpoint_url(status, 'query')
INGEST_URL = ofs_utils.endpoint_url(status, 'ingest')
print(f'Query URL:  {QUERY_URL}')
print(f'Ingest URL: {INGEST_URL}')

## 2.2 Register Entities

In [ ]:
# An Entity is a named primary key definition — it tells the Feature Store which column(s)
# to use when joining feature views to incoming scoring requests.
# join_keys must match the column names in both the stream schema and the scoring payload.
# We define 4 entities (one per velocity dimension): Customer, Merchant, DPAN, IP Address.
# Card Token velocity is derived at scoring time from DPAN features, so no separate entity needed.
customer_entity = Entity(name='FRAUD_CUSTOMER',    join_keys=['CUSTOMER_ID'])
merchant_entity = Entity(name='FRAUD_MERCHANT',    join_keys=['MERCHANT_ID'])
dpan_entity     = Entity(name='FRAUD_WALLET_DPAN', join_keys=['WALLET_DPAN'])
ip_entity       = Entity(name='FRAUD_IP_ADDRESS',  join_keys=['IP_ADDRESS'])

# Register each entity in the Feature Store (idempotent — safe to re-run).
# Entities are lightweight metadata objects; they don't store any data themselves.
for entity in [customer_entity, merchant_entity, dpan_entity, ip_entity]:
    try:
        fs.register_entity(entity)
        print(f'  Registered: {entity.name}')
    except Exception as e:
        if 'already exists' in str(e).lower():
            print(f'  Exists: {entity.name}')
        else:
            raise

## 2.3 Register Stream Source

Defines the event schema for real-time transaction ingestion. Every transaction is sent here via REST to trigger continuous velocity feature updates.

In [ ]:
from snowflake.snowpark.types import DoubleType

# A StreamSource defines the schema of the event records that will be ingested into the
# Online Feature Store. Each time a transaction is scored, a record is posted to the
# INGEST endpoint — the OFS uses these records to maintain rolling window aggregates.
#
# Only the fields needed for feature computation are included here (not the full transaction).
# AMOUNT_USD: the transaction amount — used for sum/count/max velocity features.
# IS_GBR:     1.0 if merchant country = GBR, else 0.0 — precomputed to avoid string ops.
# EVENT_TS:   the transaction timestamp — defines the time axis for all rolling windows.
#             TimestampTimeZone.NTZ = no time zone (UTC assumed, consistent with TIMESTAMP_NTZ).
txn_stream = StreamSource(
    name='FRAUD_TXN_EVENTS',
    schema=StructType([
        StructField('CUSTOMER_ID', StringType()),
        StructField('MERCHANT_ID', StringType()),
        StructField('WALLET_DPAN', StringType()),
        StructField('IP_ADDRESS',  StringType()),
        StructField('AMOUNT_USD',  DoubleType()),
        StructField('IS_GBR',      DoubleType()),
        StructField('EVENT_TS',    TimestampType(TimestampTimeZone.NTZ)),
    ]),
    desc='Real-time fraud transaction events',
)

# register_stream_source creates the underlying Snowflake stream object.
# Safe to re-run — catches 'already exists' errors gracefully.
try:
    fs.register_stream_source(txn_stream)
    print('Stream source registered: FRAUD_TXN_EVENTS')
except Exception as e:
    if 'already exists' in str(e).lower():
        print('Stream source already exists — continuing')
    else:
        raise

## 2.4 Create Stream Feature Views (CONTINUOUS Aggregation)

`FeatureAggregationMethod.CONTINUOUS` maintains running aggregates updated on every ingest event — no refresh cycle, no warehouse, < 2s end-to-end freshness.

**Why this matters:** A card-testing burst of 5 transactions spanning 30 seconds is invisible to a 33-39s refresh cycle. With CONTINUOUS aggregation, velocity counts update after each transaction — the model detects the burst from transaction 2 onwards.

> **approx_count_distinct note:** Distinct-value features (distinct merchants, DPANs) use HyperLogLog (~6.5% RSE at default precision). For fraud velocity signals this is negligible. Use `precision=14` to reduce RSE to ~0.8% if needed.

In [ ]:
import pandas as pd
from datetime import datetime

# OnlineConfig: enables this feature view in the low-latency Postgres-backed online store.
# OnlineStoreType.POSTGRES uses the managed Postgres store — sub-5ms point lookups.
online_cfg = OnlineConfig(enable=True, store_type=OnlineStoreType.POSTGRES)

def identity_transform(df: pd.DataFrame) -> pd.DataFrame:
    return df

# A single dummy row for backfill_df — StreamConfig requires at least one row to probe
# the transformation function during registration. This row is not used for actual features.
backfill_schema = StructType([
    StructField('CUSTOMER_ID', StringType()),
    StructField('MERCHANT_ID', StringType()),
    StructField('WALLET_DPAN', StringType()),
    StructField('IP_ADDRESS',  StringType()),
    StructField('AMOUNT_USD',  DoubleType()),
    StructField('IS_GBR',      DoubleType()),
    StructField('EVENT_TS',    TimestampType(TimestampTimeZone.NTZ)),
])
backfill_df = session.create_dataframe(
    [['DUMMY', 'DUMMY', 'DUMMY', '0.0.0.0', 0.0, 0.0, datetime(2020, 1, 1)]],
    schema=backfill_schema,
)

# StreamConfig links this feature view to the FRAUD_TXN_EVENTS stream source.
# transformation_fn: identity — pass events through as-is (no pre-processing needed).
# backfill_df: single dummy row — required for probe validation during registration.
stream_cfg = StreamConfig(
    stream_source=txn_stream,
    transformation_fn=identity_transform,
    backfill_df=backfill_df,
)

# ── Customer velocity (primary card-testing signal) ──────────────────────────
# This feature view computes rolling-window aggregates per CUSTOMER_ID.
# refresh_freq='1 minute'     — the feature view materialises a new snapshot every minute.
# feature_granularity='1 minute' — the smallest time bucket for aggregation.
# timestamp_col='EVENT_TS'    — the column in the stream events that defines event time.
#
# Feature.count / sum / max / min — standard aggregates over AMOUNT_USD for each window.
# Feature.approx_count_distinct — HyperLogLog-based approximate distinct count.
#   Only supported on STRING columns for Postgres online store.
#
# Time window notation: '1h', '6h', '24h', '48h', '7d' — interpreted relative to EVENT_TS.
# .alias('...') — sets the output column name in the online store and training dataset.
#
# FeatureAggregationMethod.CONTINUOUS — features are updated as each event arrives (sub-second
# freshness), not on a batch schedule. This is what enables real-time fraud detection.
customer_fv = FeatureView(
    name='FRAUD_CUSTOMER_VELOCITY',
    entities=[customer_entity],
    stream_config=stream_cfg,
    timestamp_col='EVENT_TS',
    refresh_freq='1 minute',
    feature_granularity='1 minute',
    features=[
        Feature.count( 'AMOUNT_USD',   '1h' ).alias('PURCHASES_NUM_L1H'),
        Feature.count( 'AMOUNT_USD',   '6h' ).alias('PURCHASES_NUM_L6H'),
        Feature.count( 'AMOUNT_USD',   '24h').alias('PURCHASES_NUM_L24H'),
        Feature.count( 'AMOUNT_USD',   '48h').alias('PURCHASES_NUM_L48H'),
        Feature.count( 'AMOUNT_USD',   '7d' ).alias('PURCHASES_NUM_L1WK'),
        Feature.sum(   'AMOUNT_USD',   '1h' ).alias('PURCHASES_AMT_L1H'),
        Feature.sum(   'AMOUNT_USD',   '6h' ).alias('PURCHASES_AMT_L6H'),
        Feature.sum(   'AMOUNT_USD',   '24h').alias('PURCHASES_AMT_L24H'),
        Feature.sum(   'AMOUNT_USD',   '48h').alias('PURCHASES_AMT_L48H'),
        Feature.sum(   'AMOUNT_USD',   '7d' ).alias('PURCHASES_AMT_L1WK'),
        Feature.max(   'AMOUNT_USD',   '1h' ).alias('PURCHASES_MAX_L1H'),
        Feature.max(   'AMOUNT_USD',   '24h').alias('PURCHASES_MAX_L24H'),
        Feature.max(   'AMOUNT_USD',   '7d' ).alias('PURCHASES_MAX_L1WK'),
        Feature.min(   'AMOUNT_USD',   '24h').alias('PURCHASES_MIN_L24H'),
        Feature.approx_count_distinct('MERCHANT_ID', '1h' ).alias('DISTINCT_MERCHANTS_L1H'),
        Feature.approx_count_distinct('MERCHANT_ID', '6h' ).alias('DISTINCT_MERCHANTS_L6H'),
        Feature.approx_count_distinct('MERCHANT_ID', '24h').alias('DISTINCT_MERCHANTS_L24H'),
        Feature.approx_count_distinct('MERCHANT_ID', '7d' ).alias('DISTINCT_MERCHANTS_L1WK'),
        Feature.approx_count_distinct('WALLET_DPAN', '24h').alias('DISTINCT_DPANS_L24H'),
        Feature.approx_count_distinct('WALLET_DPAN', '7d' ).alias('DISTINCT_DPANS_L1WK'),
        # AMOUNT_USD is DoubleType — approx_count_distinct only supports StringType for Postgres.
        # Use count of distinct amounts via a different approach or drop this feature.
        Feature.count('IS_GBR', '24h').alias('PURCHASES_GBR_NUM_L24H'),
    ],
    online_config=online_cfg,
    feature_aggregation_method=FeatureAggregationMethod.CONTINUOUS,
    desc='Customer velocity — CONTINUOUS, sub-2s freshness',
)
# block=True — waits for registration to complete before returning (takes 30-90s per view).
fs.register_feature_view(customer_fv, version='V1', block=True)
print('  Registered: FRAUD_CUSTOMER_VELOCITY')

In [ ]:
# ── Merchant velocity ────────────────────────────────────────────────────────
# Merchant-level features detect compromised merchants and bust-out fraud rings.
# Key signal: a sudden spike in MERCHANT_UNIQUE_CUST_L1H (many different customers
# at one merchant in 1 hour) can indicate a card-skimming or POS-compromise incident.
merchant_fv = FeatureView(
    name='FRAUD_MERCHANT_VELOCITY',
    entities=[merchant_entity],
    stream_config=stream_cfg, timestamp_col='EVENT_TS',
    refresh_freq='1 minute', feature_granularity='1 minute',
    features=[
        Feature.count( 'AMOUNT_USD',   '1h' ).alias('MERCHANT_TXN_NUM_L1H'),
        Feature.count( 'AMOUNT_USD',   '6h' ).alias('MERCHANT_TXN_NUM_L6H'),
        Feature.count( 'AMOUNT_USD',   '24h').alias('MERCHANT_TXN_NUM_L24H'),
        Feature.count( 'AMOUNT_USD',   '48h').alias('MERCHANT_TXN_NUM_L48H'),
        Feature.count( 'AMOUNT_USD',   '7d' ).alias('MERCHANT_TXN_NUM_L1WK'),
        Feature.sum(   'AMOUNT_USD',   '1h' ).alias('MERCHANT_TXN_AMT_L1H'),
        Feature.sum(   'AMOUNT_USD',   '24h').alias('MERCHANT_TXN_AMT_L24H'),
        Feature.sum(   'AMOUNT_USD',   '7d' ).alias('MERCHANT_TXN_AMT_L1WK'),
        Feature.max(   'AMOUNT_USD',   '24h').alias('MERCHANT_MAX_TXN_L24H'),
        Feature.approx_count_distinct('CUSTOMER_ID', '1h' ).alias('MERCHANT_UNIQUE_CUST_L1H'),
        Feature.approx_count_distinct('CUSTOMER_ID', '24h').alias('MERCHANT_UNIQUE_CUST_L24H'),
        Feature.approx_count_distinct('CUSTOMER_ID', '7d' ).alias('MERCHANT_UNIQUE_CUST_L1WK'),
    ],
    online_config=online_cfg,
    feature_aggregation_method=FeatureAggregationMethod.CONTINUOUS,
    desc='Merchant velocity — CONTINUOUS',
)
fs.register_feature_view(merchant_fv, version='V1', block=True)
print('  Registered: FRAUD_MERCHANT_VELOCITY')

# ── Wallet DPAN velocity ──────────────────────────────────────────────────────
# DPAN = Device Primary Account Number (the tokenised card number provisioned to a wallet).
# Key signal: DPAN_DISTINCT_CUST_L24H > 1 means the same device token is being used by
# multiple customers — a strong indicator of DPAN cloning or account sharing fraud.
dpan_fv = FeatureView(
    name='FRAUD_DPAN_VELOCITY',
    entities=[dpan_entity],
    stream_config=stream_cfg, timestamp_col='EVENT_TS',
    refresh_freq='1 minute', feature_granularity='1 minute',
    features=[
        Feature.count( 'AMOUNT_USD',   '1h' ).alias('DPAN_TXN_NUM_L1H'),
        Feature.count( 'AMOUNT_USD',   '6h' ).alias('DPAN_TXN_NUM_L6H'),
        Feature.count( 'AMOUNT_USD',   '24h').alias('DPAN_TXN_NUM_L24H'),
        Feature.count( 'AMOUNT_USD',   '48h').alias('DPAN_TXN_NUM_L48H'),
        Feature.count( 'AMOUNT_USD',   '7d' ).alias('DPAN_TXN_NUM_L1WK'),
        Feature.sum(   'AMOUNT_USD',   '24h').alias('DPAN_TXN_AMT_L24H'),
        Feature.sum(   'AMOUNT_USD',   '7d' ).alias('DPAN_TXN_AMT_L1WK'),
        Feature.approx_count_distinct('CUSTOMER_ID', '24h').alias('DPAN_DISTINCT_CUST_L24H'),
        Feature.approx_count_distinct('CUSTOMER_ID', '7d' ).alias('DPAN_DISTINCT_CUST_L1WK'),
    ],
    online_config=online_cfg,
    feature_aggregation_method=FeatureAggregationMethod.CONTINUOUS,
    desc='DPAN velocity — shared DPAN = fraud signal',
)
fs.register_feature_view(dpan_fv, version='V1', block=True)
print('  Registered: FRAUD_DPAN_VELOCITY')

# ── IP Address velocity ───────────────────────────────────────────────────────
# IP-level features detect botnet traffic and proxy/VPN fraud rings.
# Key signal: IP_DISTINCT_CUST_L1H > threshold means many different customers
# are transacting from the same IP in a short window — typical of a bot farm or NAT proxy.
ip_fv = FeatureView(
    name='FRAUD_IP_VELOCITY',
    entities=[ip_entity],
    stream_config=stream_cfg, timestamp_col='EVENT_TS',
    refresh_freq='1 minute', feature_granularity='1 minute',
    features=[
        Feature.count( 'AMOUNT_USD',   '1h' ).alias('IP_TXN_NUM_L1H'),
        Feature.count( 'AMOUNT_USD',   '6h' ).alias('IP_TXN_NUM_L6H'),
        Feature.count( 'AMOUNT_USD',   '24h').alias('IP_TXN_NUM_L24H'),
        Feature.count( 'AMOUNT_USD',   '48h').alias('IP_TXN_NUM_L48H'),
        Feature.count( 'AMOUNT_USD',   '7d' ).alias('IP_TXN_NUM_L1WK'),
        Feature.sum(   'AMOUNT_USD',   '24h').alias('IP_TXN_AMT_L24H'),
        Feature.sum(   'AMOUNT_USD',   '7d' ).alias('IP_TXN_AMT_L1WK'),
        Feature.approx_count_distinct('CUSTOMER_ID', '1h' ).alias('IP_DISTINCT_CUST_L1H'),
        Feature.approx_count_distinct('CUSTOMER_ID', '24h').alias('IP_DISTINCT_CUST_L24H'),
        Feature.approx_count_distinct('CUSTOMER_ID', '7d' ).alias('IP_DISTINCT_CUST_L1WK'),
    ],
    online_config=online_cfg,
    feature_aggregation_method=FeatureAggregationMethod.CONTINUOUS,
    desc='IP velocity — shared IP = bot farm signal',
)
fs.register_feature_view(ip_fv, version='V1', block=True)
print('  Registered: FRAUD_IP_VELOCITY')
print('\nAll 4 entity stream feature views registered.')

## 2.5 Customer Profile Feature View (Batch, Daily Refresh)

Static lifetime features that change slowly. Served from a DT-backed batch online feature view refreshed daily.

In [ ]:
# The customer profile feature view is a BATCH feature view (not CONTINUOUS/stream-based).
# It reads from a pre-aggregated table (CUSTOMER_PROFILE) that is refreshed daily.
# Batch feature views are appropriate for slowly-changing attributes like:
#   - Lifetime transaction count and total spend
#   - Account age in days
#   - Average transaction amount (30-day rolling)
# These features don't need sub-second freshness — daily refresh is sufficient.

# CUSTOMER_PROFILE doesn't exist as a table — build it by joining DIM_CUSTOMERS
# (static attributes) with aggregates from FRAUD_TRANSACTIONS.
session.sql('''
    CREATE TABLE IF NOT EXISTS FRAUD_DEMO_DEV.TRANSACTIONS.CUSTOMER_PROFILE AS
    SELECT
        c.CUSTOMER_ID,
        c.CUSTOMER_AGE,
        c.DAYS_SINCE_REGISTRATION,
        c.IS_HIGH_RISK,
        COALESCE(t.LIFETIME_TXN_COUNT, 0)    AS LIFETIME_TXN_COUNT,
        COALESCE(t.LIFETIME_TXN_TOTAL, 0.0)  AS LIFETIME_TXN_TOTAL,
        COALESCE(t.AVG_TXN_AMOUNT_30D, 0.0)  AS AVG_TXN_AMOUNT_30D
    FROM FRAUD_DEMO_DEV.TRANSACTIONS.DIM_CUSTOMERS c
    LEFT JOIN (
        SELECT
            CUSTOMER_ID,
            COUNT(*)                          AS LIFETIME_TXN_COUNT,
            SUM(PURCHASE_AMOUNT)              AS LIFETIME_TXN_TOTAL,
            AVG(CASE WHEN TRANSACTION_TS >= DATEADD(DAY, -30, CURRENT_TIMESTAMP())
                     THEN PURCHASE_AMOUNT END) AS AVG_TXN_AMOUNT_30D
        FROM FRAUD_DEMO_DEV.TRANSACTIONS.FRAUD_TRANSACTIONS
        GROUP BY CUSTOMER_ID
    ) t ON c.CUSTOMER_ID = t.CUSTOMER_ID
''').collect()
print('Created: FRAUD_DEMO_DEV.TRANSACTIONS.CUSTOMER_PROFILE')

profile_src = session.table('FRAUD_DEMO_DEV.TRANSACTIONS.CUSTOMER_PROFILE')

profile_fv = FeatureView(
    name='FRAUD_CUSTOMER_PROFILE',
    entities=[customer_entity],
    feature_df=profile_src,
    refresh_freq='1 day',
    online_config=OnlineConfig(enable=True, target_lag='1h', store_type=OnlineStoreType.POSTGRES),
    desc='Customer profile: lifetime stats, account age. Batch online FV, daily refresh.',
)
fs.register_feature_view(profile_fv, version='V1', block=True)
print('Registered: FRAUD_CUSTOMER_PROFILE (batch, daily refresh)')

## 2.6 Freshness Benchmark (RUN LIVE)

Measures time from REST ingest to feature visible at query time. Target: < 2 seconds.

In [ ]:
# Freshness benchmark: measures how quickly an ingested event propagates to ALL 4 feature views.
# A single transaction fans out to 4 independent entity pipelines (customer, merchant, DPAN, IP).
# The "time to score" = max latency across all 4, since scoring can't proceed until all are fresh.

from datetime import datetime, timezone
import concurrent.futures

print('='*60)
print('FRESHNESS BENCHMARK')
print('='*60)
print('''
What this measures:
  A single transaction event is ingested via the OFS HTTP endpoint.
  The event fans out to 4 independent CONTINUOUS feature view pipelines
  (Customer, Merchant, DPAN, IP). We poll each until its count feature
  increments from 0 → 1.

  Latency = time from ingest response to first query showing the update.
  "Time-to-score" = slowest of the 4 (all must be ready before scoring).
  Target: < 2 seconds.
''')

# The 4 feature views and their entity key columns
FV_ENTITIES = [
    ('FRAUD_CUSTOMER_VELOCITY', 'CUSTOMER_ID'),
    ('FRAUD_MERCHANT_VELOCITY', 'MERCHANT_ID'),
    ('FRAUD_DPAN_VELOCITY',     'WALLET_DPAN'),
    ('FRAUD_IP_VELOCITY',       'IP_ADDRESS'),
]

# Generate unique entity keys for this test run
test_cust = f'CUST_{random.randint(500000, 599999)}'
test_merch = f'MERCH_{random.randint(500000, 599999)}'
test_dpan = f'DPAN_{random.randint(500000, 599999)}'
test_ip = f'IP_{random.randint(500000, 599999)}'
entity_keys = {
    'CUSTOMER_ID': test_cust,
    'MERCHANT_ID': test_merch,
    'WALLET_DPAN': test_dpan,
    'IP_ADDRESS':  test_ip,
}
# Each FV uses a count feature — we check if it increments from 0 to 1
FEATURE_PER_FV = {
    'FRAUD_CUSTOMER_VELOCITY': 'PURCHASES_NUM_L1H',
    'FRAUD_MERCHANT_VELOCITY': 'MERCHANT_TXN_NUM_L1H',
    'FRAUD_DPAN_VELOCITY':     'DPAN_TXN_NUM_L1H',
    'FRAUD_IP_VELOCITY':       'IP_TXN_NUM_L1H',
}

print(f'Test entities: {test_cust}, {test_merch}, {test_dpan}, {test_ip}')

# Step 1: Verify baselines are all 0 (fresh keys never seen before)
print('\n--- Baselines (expect all 0 for unseen keys) ---')
for fv_name, key_col in FV_ENTITIES:
    r = requests.post(f'{QUERY_URL}/api/v1/query', headers=HEADERS, json={
        'name': fv_name, 'version': 'V1', 'object_type': 'feature_view',
        'request_rows': [{'entity': {key_col: entity_keys[key_col]}}],
        'features': [FEATURE_PER_FV[fv_name]],
    }, timeout=10)
    val = r.json().get('results', [{}])[0].get('features', [0])[0] or 0
    print(f'  {fv_name}: {FEATURE_PER_FV[fv_name]} = {val}')

# Step 2: Ingest a single transaction event
print('\n--- Ingest single event ---')
ingest_ts = datetime.now(timezone.utc)
ir = requests.post(f'{INGEST_URL}/api/v1/ingest', headers=HEADERS, json={
    'dry_run': False,
    'records': {'FRAUD_TXN_EVENTS': [{
        'CUSTOMER_ID': test_cust, 'MERCHANT_ID': test_merch,
        'WALLET_DPAN': test_dpan, 'IP_ADDRESS': test_ip,
        'AMOUNT_USD': round(random.uniform(10, 500), 2), 'IS_GBR': 1.0,
        'EVENT_TS': ingest_ts.strftime('%Y-%m-%dT%H:%M:%SZ'),
    }]},
}, timeout=10)
print(f'  HTTP {ir.status_code}')
start = time.time()

# Step 3: Poll all 4 feature views until each updates
print('\n--- Polling (250ms intervals, 30s timeout) ---')
latencies = {}  # fv_name → latency_ms
pending = set(fv for fv, _ in FV_ENTITIES)

for i in range(120):  # 30s max (120 × 250ms)
    time.sleep(0.25)
    for fv_name, key_col in FV_ENTITIES:
        if fv_name not in pending:
            continue
        r = requests.post(f'{QUERY_URL}/api/v1/query', headers=HEADERS, json={
            'name': fv_name, 'version': 'V1', 'object_type': 'feature_view',
            'request_rows': [{'entity': {key_col: entity_keys[key_col]}}],
            'features': [FEATURE_PER_FV[fv_name]],
        }, timeout=10)
        if r.status_code == 200:
            results = r.json().get('results', [])
            if results and (results[0].get('features', [0])[0] or 0) > 0:
                ms = (time.time() - start) * 1000
                latencies[fv_name] = ms
                pending.discard(fv_name)
                print(f'  {fv_name}: {ms:.0f}ms')
    if not pending:
        break
    if i % 8 == 0 and i > 0:
        print(f'  [{i*0.25:.1f}s] still waiting on: {", ".join(pending)}')

# Step 4: Summary
print('\n--- Results ---')
print('  Metric: end-to-end freshness (ingest response → feature queryable)')
if pending:
    print(f'  TIMEOUT: {pending} did not update within 30s')
if latencies:
    for fv, ms in sorted(latencies.items(), key=lambda x: x[1]):
        print(f'  {fv}: {ms:.0f}ms')
    print(f'\n  Fastest (single pipeline):  {min(latencies.values()):.0f}ms')
    print(f'  Slowest (time-to-score):    {max(latencies.values()):.0f}ms')
    print(f'  Verdict: {max(latencies.values()):.0f}ms {"< 2s PASS" if max(latencies.values()) < 2000 else "> 2s FAIL"}')

## 2.7 Latency Benchmark

10 warmup + 200 measured requests. Single-entity and 4-entity concurrent (per-transaction) lookups using real entity keys.

In [ ]:
print('='*60)
print('QUERY LATENCY BENCHMARK')
print('='*60)
print('''
What this measures:
  The HTTP round-trip time to READ features from the Online Feature Store.
  No data is ingested — we are purely measuring how fast the Postgres-backed
  online store responds to point lookups by entity key.

How it differs from the Freshness Benchmark:
  • Freshness = time for a NEW event to propagate through the CONTINUOUS
    pipeline and become queryable (write path → aggregation → read path).
    It also queries all 4 feature views, but it measures the WRITE PIPELINE
    delay — how long until freshly ingested data is visible.
  • Latency = time to READ already-computed features (read path only).
    No ingestion occurs. Features already exist in the online store.
    This isolates the Postgres lookup + HTTP overhead from pipeline processing.

  In production, total scoring time = freshness + latency:
    - Freshness tells you how stale features might be at query time.
    - Latency tells you how long the scoring service blocks waiting for
      the feature vector response.

Benchmark 1: Single-entity lookup (customer only)
  → Measures raw OFS query overhead for one feature view.

Benchmark 2: 4-entity concurrent lookup (customer + merchant + DPAN + IP)
  → Simulates a real scoring call: 4 lookups fired in parallel.
  → Reports wall-clock time = max(4 individual latencies).
  → This is what the fraud model waits for per transaction.
''')

N_WARMUP, N_REQUESTS = 10, 200

# Sample 250 real entity keys from the transactions table to use as test inputs.
# Using real keys (rather than random strings) ensures cache behaviour is realistic.
samples = session.sql('SELECT CUSTOMER_ID, MERCHANT_ID, WALLET_DPAN, IP_ADDRESS FROM FRAUD_DEMO_DEV.TRANSACTIONS.FRAUD_TRANSACTIONS SAMPLE (250 ROWS)').collect()
print(f'Sampled {len(samples)} entity keys\n')

def qlat(name, key_col, key_val):
    """Issue a single feature view query and return the round-trip latency in ms."""
    t0 = time.perf_counter()
    r = requests.post(f'{QUERY_URL}/api/v1/query', headers=HEADERS, json={
        'name': name, 'version': 'V1', 'object_type': 'feature_view',
        'request_rows': [{'entity': {key_col: key_val}}],
    }, timeout=10)
    r.raise_for_status()
    return (time.perf_counter() - t0) * 1000

# Benchmark 1: single entity
print('='*55); print('BENCHMARK 1: Single-entity (customer)'); print('='*55)
print('  Metric: HTTP round-trip for one feature view query')
lat_s = []
for i in range(N_WARMUP + N_REQUESTS):
    row = samples[i % len(samples)]
    lat = qlat('FRAUD_CUSTOMER_VELOCITY', 'CUSTOMER_ID', row['CUSTOMER_ID'])
    if i >= N_WARMUP: lat_s.append(lat)
    if i == N_WARMUP - 1: print('  Warmup complete (first 10 requests discarded)')
arr_s = np.array(lat_s)
print(f'  n={N_REQUESTS} requests')
print(f'  p50: {np.percentile(arr_s,50):.1f}ms  p95: {np.percentile(arr_s,95):.1f}ms  p99: {np.percentile(arr_s,99):.1f}ms')

# Benchmark 2: 4-entity concurrent
print(f'\n{"="*55}'); print('BENCHMARK 2: 4-entity concurrent (per-transaction)'); print('='*55)
print('  Metric: wall-clock time for 4 parallel feature view queries')
def q_all(row):
    """Fire 4 entity lookups in parallel; return the wall-clock time (= slowest of the 4)."""
    with concurrent.futures.ThreadPoolExecutor(max_workers=4) as ex:
        fs_ = [ex.submit(qlat, 'FRAUD_CUSTOMER_VELOCITY', 'CUSTOMER_ID', row['CUSTOMER_ID']),
               ex.submit(qlat, 'FRAUD_MERCHANT_VELOCITY', 'MERCHANT_ID', row['MERCHANT_ID']),
               ex.submit(qlat, 'FRAUD_DPAN_VELOCITY',     'WALLET_DPAN',  row['WALLET_DPAN']),
               ex.submit(qlat, 'FRAUD_IP_VELOCITY',       'IP_ADDRESS',   row['IP_ADDRESS'])]
    return max(f.result() for f in fs_)

lat_m = []
for i in range(N_WARMUP + N_REQUESTS):
    row = samples[i % len(samples)]
    wall = q_all(row)
    if i >= N_WARMUP: lat_m.append(wall)
    if i == N_WARMUP - 1: print('  Warmup complete (first 10 requests discarded)')
arr_m = np.array(lat_m)
print(f'  n={N_REQUESTS} requests')
print(f'  p50: {np.percentile(arr_m,50):.1f}ms  p95: {np.percentile(arr_m,95):.1f}ms  p99: {np.percentile(arr_m,99):.1f}ms')
print(f'\n  → This is the per-transaction feature fetch overhead in production.')

---
## Summary

| Component | Freshness | Method |
|---|---|---|
| FRAUD_CUSTOMER_VELOCITY | < 2 seconds | CONTINUOUS stream aggregation |
| FRAUD_MERCHANT_VELOCITY | < 2 seconds | CONTINUOUS stream aggregation |
| FRAUD_DPAN_VELOCITY | < 2 seconds | CONTINUOUS stream aggregation |
| FRAUD_IP_VELOCITY | < 2 seconds | CONTINUOUS stream aggregation |
| FRAUD_CUSTOMER_PROFILE | Daily | Batch online FV |

**Online Service pricing** depends on entity cardinality, feature view count, ingest rate, and time-window depth. No 24/7 DT warehouse required.

**Next:** NB03 — train XGBoost from the transactions table.